# Chaining Graphs

Use the routed profile runtime to see how an entry graph hands state to tail graphs.

This notebook is about structural intuition:
- what the entry graph owns
- which tail graph a pathway selects
- what state is inherited into the tail graph
- what final treatment endpoint comes out


In [1]:
from pathlib import Path
import sys
import random

import pandas as pd
from IPython.display import HTML, Markdown, display

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

from scoop.configuration.config_loader import load_profile_manifest, load_routed_profile
from scoop.utils.document_generation import make_mermaid, mermaid_html


2026-04-06 21:59:57,457 | DEBUG   | scoop | SCOOP logging initialized


In [2]:
profile_id = "nsclc"
runtime = load_routed_profile(profile_id, base_dir=CONFIG_ROOT)
manifest = load_profile_manifest(profile_id, base_dir=CONFIG_ROOT)

display(Markdown(f"## Routed Runtime For `{profile_id}`"))
display(pd.DataFrame([{
    "profile_id": manifest.profile_id,
    "default_graph": manifest.default_graph,
    "entry_graph": runtime.entry_graph_name,
    "tail_graph_count": len(runtime.tail_contexts),
}]))

display(Markdown("## Pathway To Tail Graph Map"))
display(pd.DataFrame([
    {"pathway": pathway, "tail_graph": graph_name}
    for pathway, graph_name in runtime.pathway_to_graph.items()
]).sort_values(["pathway"]).reset_index(drop=True))


## Routed Runtime For `nsclc`

,profile_id,default_graph,entry_graph,tail_graph_count
0,nsclc,nsclc,nsclc,5


## Pathway To Tail Graph Map

,pathway,tail_graph
0,nsclc_stage1,nsclc_stg1
1,nsclc_stage23_resectable,nsclc_stg23_resectable
2,nsclc_stage3_unresectable,nsclc_stg3_unresectable
3,nsclc_stage4_driver,nsclc_stg4_driver
4,nsclc_stage4_nodrivers,nsclc_stg4_nodrivers


In [3]:
def graph_overview_rows(runtime):
    rows = []
    entry = runtime.entry_context.graph
    rows.append({
        "graph_name": runtime.entry_graph_name,
        "role": "entry",
        "node_count": len(entry.nodes),
        "branch_nodes": sum(node.type == "branch" for node in entry.nodes.values()),
        "active_when_nodes": sum(bool(getattr(node, "active_when", None)) for node in entry.nodes.values()),
        "treatment_nodes": sum(name.startswith("treatment_episode.") for name in entry.nodes),
    })
    for name, ctx in runtime.tail_contexts.items():
        graph = ctx.graph
        rows.append({
            "graph_name": name,
            "role": "tail",
            "node_count": len(graph.nodes),
            "branch_nodes": sum(node.type == "branch" for node in graph.nodes.values()),
            "active_when_nodes": sum(bool(getattr(node, "active_when", None)) for node in graph.nodes.values()),
            "treatment_nodes": sum(name.startswith("treatment_episode.") for name in graph.nodes),
        })
    return pd.DataFrame(rows).sort_values(["role", "graph_name"]).reset_index(drop=True)

display(Markdown("## Graph Inventory"))
display(graph_overview_rows(runtime))


## Graph Inventory

,graph_name,role,node_count,branch_nodes,active_when_nodes,treatment_nodes
0,nsclc,entry,32,6,0,0
1,nsclc_stg1,tail,6,1,4,5
2,nsclc_stg23_resectable,tail,8,1,6,6
3,nsclc_stg3_unresectable,tail,10,3,7,7
4,nsclc_stg4_driver,tail,8,0,8,8
5,nsclc_stg4_nodrivers,tail,11,0,10,10


In [4]:
selected_pathway = "nsclc_stage23_resectable"
selected_tail_graph = runtime.pathway_to_graph[selected_pathway]

display(Markdown(f"## Entry Graph: `{runtime.entry_graph_name}`"))
HTML(runtime.entry_context.graph.to_mermaid())


## Entry Graph: `nsclc`

In [5]:

display(Markdown(f"## Tail Graph For `{selected_pathway}`: `{selected_tail_graph}`"))
HTML(runtime.tail_contexts[selected_tail_graph].graph.to_mermaid())

## Tail Graph For `nsclc_stage23_resectable`: `nsclc_stg23_resectable`

In [6]:
random.seed(17)
for _ in range(5000):
    routed = runtime.sample_routed_patient()
    if routed.pathway == selected_pathway:
        break
else:
    raise RuntimeError(f"Did not sample pathway {selected_pathway!r} within 5000 draws")

display(Markdown("## One Entry -> Tail Sample"))
display(pd.DataFrame([{
    "pathway": routed.pathway,
    "tail_graph_name": routed.tail_graph_name,
    "primary_category_id": routed.graph_outcome.primary_category_id if routed.graph_outcome else None,
    "active_endpoints": routed.graph_outcome.active_endpoints if routed.graph_outcome else tuple(),
    "outcome_error": routed.outcome_error,
}]))

comparison_keys = [
    key for key in routed.final_sample
    if key.startswith("person.") or key.startswith("condition_occurrence.") or key.startswith("condition_episode.") or key.startswith("treatment_episode.")
]
comparison = pd.DataFrame([
    {
        "field": key,
        "entry_value": routed.entry_sample.get(key),
        "final_value": routed.final_sample.get(key),
        "changed_in_tail": routed.entry_sample.get(key) != routed.final_sample.get(key),
    }
    for key in comparison_keys
    if routed.final_sample.get(key) is not None or routed.entry_sample.get(key) is not None
])
display(comparison)


## One Entry -> Tail Sample

,pathway,tail_graph_name,primary_category_id,active_endpoints,outcome_error
0,nsclc_stage23_resectable,nsclc_stg23_resectable,neoadjuvant_chemo_ici,"(neoadjuvant_chemo_ici,)",None


,field,entry_value,final_value,changed_in_tail
0,person.age,57,57,False
1,person.year_of_diagnosis,2019,2019,False
2,person.years_since_diagnosis,0,0,False
3,person.sex,female,female,False
4,person.smoking_history,former,former,False
5,person.weight,76.255539,76.255539,False
6,condition_occurrence.laterality,right,right,False
7,condition_occurrence.n_stage,N0,N0,False
8,condition_occurrence.t_8th,T1a,T1a,False
9,condition_occurrence.m_stage,M0,M0,False
